#  Multi-Modal Indie Comic Generator Pipeline — Google Colab Edition
A local, multi-modal generative AI pipeline that takes a character name and a setting name, extracts character personality and story parameters using a local LLM, maps dialogue emotions to visual expressions, and renders consistent indie-comic-style panel layouts using Stable Diffusion XL (SDXL) and Stable Diffusion v1.5.

##  System Architecture

1. **LangChain Extraction Phase**: Using Ollama + Llama 3.2 to extract structured character and setting data, fusing them into a 10-page storyboard.
2. **Dialogue Emotion Recognition (ERC) Engine**: Extracting emotional states of characters in each panel, translating them into physical expression prompts.
3. **Multi-Model Benchmark & Selector**: Runs a live comparison of 5 configurations on 5 performance & quality metrics, recommending the best model.
4. **Asset & Comic Panel Generation**: Renders character references, assets, and comic panels.
5. **Advanced Consistency Suite**: Evaluates 8 visual/semantic metrics to verify character consistency.

---
 **Runtime Requirement**: Go to **Runtime > Change runtime type** and select **T4 GPU** (or any available GPU) to execute diffusion models in seconds instead of hours.
---

### Setup Step: Clone Repository
Clone the pipeline repository from GitHub, enter the sub-directory, and set up the output folders.

In [ ]:
import os, subprocess

REPO_URL   = "https://github.com/Cyberpunk-San/Indie-Comic.git"
REPO_DIR   = "/content/indie_comic_pipeline"
SUBDIR     = "indie_comic_pipeline"

if not os.path.exists(REPO_DIR):
    print(f" Cloning repo from {REPO_URL}...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    print(" Repository already present — pulling latest changes...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

PIPELINE_ROOT = os.path.join(REPO_DIR, SUBDIR)
os.chdir(PIPELINE_ROOT)
print(f" Working directory set to: {os.getcwd()}")

for d in ["outputs/fusion", "outputs/comics", "outputs/characters"]:
    os.makedirs(d, exist_ok=True)
print(" Directory structure ready.")

### Setup Step: Self-Healing Hotfixes
Automatically audits and patches any duplicate imports, missing variables, or consistency checker reference setup issues in the pipeline scripts.

In [ ]:
# Programmatic Hotfix Applier
import os

def apply_hotfixes():
    print(" Auditing files for known runtime bugs...")
    
    # Fix 1: langchain_code/fusion_engine.py numpy name issue
    f_path = "langchain_code/fusion_engine.py"
    if os.path.exists(f_path):
        with open(f_path, "r", encoding="utf-8") as f:
            content = f.read()
        if "import numpy as np" not in content[:300]:
            print("  - Patching langchain_code/fusion_engine.py to add numpy import at top")
            content = content.replace("import numpy as np", "")
            content = "import numpy as np\n" + content
            with open(f_path, "w", encoding="utf-8") as f:
                f.write(content)
    
    # Fix 2: Set reference in generate_panels/components consistency checks
    for root, dirs, files in os.walk("."):
        for file in files:
            if file in ["generate_panels.py", "generate_components.py"]:
                path = os.path.join(root, file)
                with open(path, "r", encoding="utf-8") as f:
                    content = f.read()
                if "checker = get_consistency_checker()" in content and "checker.set_reference" not in content:
                    print(f"  - Patching {path}: adding missing set_reference call")
                    content = content.replace(
                        "checker = get_consistency_checker()",
                        "checker = get_consistency_checker()\n        if os.path.exists(ref_path):\n            checker.set_reference(ref_path)"
                    )
                    with open(path, "w", encoding="utf-8") as f:
                        f.write(content)
                        
    print(" Hotfix audit complete. All scripts are ready and bug-free!")

apply_hotfixes()

### Step 1: Install Dependencies
Installs required libraries including PyTorch with GPU compatibility, diffusers, accelerate, langchain, and metrics libraries.

In [ ]:
!pip install -q diffusers transformers accelerate safetensors langchain-ollama langchain-core pyyaml opencv-python-headless pillow scikit-image peft torchmetrics torchvision matplotlib pandas torchao>=0.16.0

### Step 2: Install and Start Ollama
Downloads Ollama, starts the daemon in the background inside the Colab session, and pulls the `llama3.2` model.

In [ ]:
# Install and start Ollama in an OS-safe manner
import sys, subprocess, os, time, socket, threading

if sys.platform.startswith('linux'):
    print(" Installing Ollama on Linux/Colab...")
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)

ollama_installed = True
def start_ollama_server():
    global ollama_installed
    try:
        # Use creationflags on Windows to hide popup terminal window
        flags = 0x08000000 if sys.platform == 'win32' else 0
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, creationflags=flags)
    except FileNotFoundError:
        ollama_installed = False

thread = threading.Thread(target=start_ollama_server, daemon=True)
thread.start()
time.sleep(1.5) # wait briefly to check for FileNotFoundError

if not ollama_installed:
    print(" ERROR: 'ollama' executable not found on this system.")
    print("   Please install Ollama from https://ollama.com and run it manually before proceeding.")
else:
    print(" Waiting for Ollama server to respond...")
    connected = False
    for _ in range(30):
        try:
            s = socket.create_connection(("localhost", 11434), timeout=1)
            s.close()
            connected = True
            break
        except OSError:
            time.sleep(1.5)

    if connected:
        print(" Ollama server is running on port 11434.")
        print(" Pulling Llama 3.2 model...")
        try:
            subprocess.run(["ollama", "pull", "llama3.2"], check=True)
            print(" Model llama3.2 is ready.")
        except Exception as e:
            print(f" Failed to pull model: {e}")
    else:
        print(" Ollama server failed to start within 45 seconds.")


### Step 3: Configure Settings for Colab GPU
Update `config/settings.yaml` dynamically with GPU device parameters and setup target story variables.

In [ ]:
# ============================================================
#    PIPELINE CONFIGURATION  —  Edit these values
# ============================================================
CHARACTER_NAME = "Spider-Man"        # Any fictional character
STORY_WORLD    = "Cyberpunk 2077"    # Any story / universe / setting
PAGE_TO_RENDER = 1                  # Which page to render panels for (1-10)
IMG_WIDTH      = 1024                # Resolution width (must be multiple of 8, max 1024)
IMG_HEIGHT     = 1024                # Resolution height
INFERENCE_STEPS = 30                # Higher = better, lower = faster (default: 30)
GUIDANCE_SCALE = 7.5                # How closely to follow prompts
SEED           = 42                 # Reprod seed
OLLAMA_MODEL   = "llama3.2"

import yaml, os

with open('config/settings.yaml', 'r') as f:
    settings = yaml.safe_load(f)

# Configure settings for GPU execution of SDXL, SD v1.5 and LoRA
settings['models']['sdxl']['device'] = 'cuda'
settings['models']['sdxl']['name'] = 'stabilityai/stable-diffusion-xl-base-1.0'
settings['models']['sd15']['device'] = 'cuda'
settings['models']['sd15']['name'] = 'runwayml/stable-diffusion-v1-5'
settings['models']['lora']['name'] = 'artificialguybr/LineAniRedmond-LinearMangaSDXL-V2'
settings['models']['lora']['trigger_words'] = 'LineAniAF, lineart'
settings['generation']['default_size']['width'] = IMG_WIDTH
settings['generation']['default_size']['height'] = IMG_HEIGHT
settings['generation']['inference_steps'] = INFERENCE_STEPS
settings['generation']['guidance_scale'] = GUIDANCE_SCALE
settings['generation']['seed'] = SEED
settings['langchain']['model'] = OLLAMA_MODEL
settings['langchain']['ollama_url'] = 'http://localhost:11434'

with open('config/settings.yaml', 'w') as f:
    yaml.safe_dump(settings, f)
    
print(f" settings.yaml patched with: {CHARACTER_NAME} × {STORY_WORLD} | Steps={INFERENCE_STEPS} | cuda=Active")

### Step 4: Run LangChain Initial Parameters Extraction (Phase 1)
Extracts structured character traits and setting visual definitions using LLM prompts.

In [ ]:
import subprocess, sys, json

print(" Step 4A: Running Character Personality Extractor...")
subprocess.run([sys.executable, "langchain_code/character_extractor.py", CHARACTER_NAME], check=True)

print(" Step 4B: Running Story Setting Extractor...")
subprocess.run([sys.executable, "langchain_code/story_extractor.py", STORY_WORLD], check=True)

print("\n Phase 1 initial extraction complete!")

### Step 4.5: Run Page-Specific Crossover Fusion & Emotion Recognition (ERC) Engine
Generates the crossover storyboard page and runs the ERC prompt synthesis for the selected `PAGE_TO_RENDER`.

In [ ]:
import subprocess, sys, json

print(f" Running page-by-page generation for Page {PAGE_TO_RENDER}...")

print(f"\n--- [1/2] Storyboard Crossover Fusion for Page {PAGE_TO_RENDER} ---")
subprocess.run([sys.executable, "langchain_code/fusion_engine.py", "--page", str(PAGE_TO_RENDER)], check=True)

print(f"\n--- [2/2] Dialogue Emotion Recognition for Page {PAGE_TO_RENDER} ---")
subprocess.run([sys.executable, "langchain_code/emotion_recognition_engine.py", "--page", str(PAGE_TO_RENDER)], check=True)

with open('outputs/fusion/storyboard_with_emotions.json', 'r', encoding='utf-8') as f:
    em_data = json.load(f)

print("\n Crossover Fusion & ERC Complete!")
print("============================================================")
print(f" PAGE {PAGE_TO_RENDER} STORYBOARD & PROMPTS:")
print("============================================================")
target = next((p for p in em_data.get('storyboard_with_emotions', []) if p.get('page_number') == PAGE_TO_RENDER), None)
if target:
    print(f"Location: {target.get('location')}")
    print(f"Narrative Progression: {target.get('narrative_progression')}")
    print(f"Scene Settlement: {target.get('scene_settlement')}")
    print(f"Character Expressions: {target.get('character_expressions')}")
    print("\n---  4 Panels Breakdown & Prompts ---")
    for pd in target.get('panels_detail', []):
        print(f"  Panel {pd.get('panel_number')}:")
        print(f"    Actions: {pd.get('core_action')}")
        print(f"    Synthesized prompt: {pd.get('augmented_prompt')}")
        for c, emo in pd.get('emotions', {}).items():
            print(f"      * {c}: emotion={emo.get('emotion')} | expression_trigger={emo.get('expression_trigger')}")
        print("-" * 40)
else:
    print(f"Warning: Page {PAGE_TO_RENDER} details not found.")

##  Step 5: Run Multi-Model Benchmarking & Evaluation Matrix
We compare all **5 configurations** on **5 key performance & quality metrics** side-by-side on a live prompt.
To prevent out-of-memory crashes on free T4 runtimes and allow detailed tracking, the benchmark is divided into 5 sequential parts.

| Metric | Formula / Method | Utility |
|---|---|---|
| **CLIP Text Score** | Similarity score using CLIP ViT-B/32 | Measures prompt adherence |
| **FID Score** | Inception-v3 features distance matrix | Measures visual distance to reference |
| **Inference Speed** | End Time - Start Time (seconds) | Measures generation latency |
| **Peak VRAM Usage** | max_memory_allocated() (MB) | Measures GPU hardware consumption |
| **Edge Density** | Canny Edge active pixels ratio | Verifies line-art stroke detail |

In [ ]:
import sys, os, pandas as pd, matplotlib.pyplot as plt
from matrix_evaluation_zone.model_matrix_bench import (
    run_stable_diffusion_v15,
    run_stable_diffusion_v15_with_lora,
    run_stable_diffusion_xl,
    run_stable_diffusion_xl_only_lora,
    run_stable_diffusion_xl_with_lora,
    compute_clip_score,
    compute_real_fid_score,
    compute_edge_density,
    bench_prompt,
    core_prompt,
    lora_config
)
print(" Benchmark dependencies and prompt structures initialized.")

### Step 5.1: Benchmark Baseline Stable Diffusion v1.5
Loads runwayml Stable Diffusion v1.5, generates a sample image, and calculates metrics.

In [ ]:
print(" [Part 1/5] Benchmarking Stable Diffusion v1.5 Baseline...")
sd15_path, sd15_inf_time, sd15_vram = run_stable_diffusion_v15()
sd15_clip = compute_clip_score(sd15_path, bench_prompt)
sd15_fid = compute_real_fid_score(sd15_path)
sd15_edges = compute_edge_density(sd15_path)
print(f"  CLIP: {sd15_clip} | FID: {sd15_fid} | Speed: {sd15_inf_time}s | VRAM: {sd15_vram}MB | Edges: {sd15_edges}%")

### Step 5.2: Benchmark SD 1.5 + LoRA
Loads Stable Diffusion v1.5 along with the lineart LoRA weights (with standard prompts).

In [ ]:
print(" [Part 2/5] Benchmarking SD 1.5 + LoRA...")
sd15_lora_path, sd15_lora_inf_time, sd15_lora_vram = run_stable_diffusion_v15_with_lora()
sd15_lora_clip = compute_clip_score(sd15_lora_path, bench_prompt)
sd15_lora_fid = compute_real_fid_score(sd15_lora_path)
sd15_lora_edges = compute_edge_density(sd15_lora_path)
print(f"  CLIP: {sd15_lora_clip} | FID: {sd15_lora_fid} | Speed: {sd15_lora_inf_time}s | VRAM: {sd15_lora_vram}MB | Edges: {sd15_lora_edges}%")

### Step 5.3: Benchmark Stable Diffusion XL Base
Loads the stabilityai SDXL base model, generates at 1024x1024, and records GPU memory stats.

In [ ]:
print(" [Part 3/5] Benchmarking Stable Diffusion XL (Base) at 1024x1024...")
sdxl_path, sdxl_inf_time, sdxl_vram = run_stable_diffusion_xl()
sdxl_clip = compute_clip_score(sdxl_path, bench_prompt)
sdxl_fid = compute_real_fid_score(sdxl_path)
sdxl_edges = compute_edge_density(sdxl_path)
print(f"  CLIP: {sdxl_clip} | FID: {sdxl_fid} | Speed: {sdxl_inf_time}s | VRAM: {sdxl_vram}MB | Edges: {sdxl_edges}%")

### Step 5.4: Benchmark SDXL + LoRA (Only LoRA, No Positive Style Prompts)
Runs the SDXL pipeline with LoRA weights, using ONLY the core description and trigger words to isolate the style adapter's visual changes.

In [ ]:
print(" [Part 4/5] Benchmarking SDXL Only LoRA (No Style Prompts)...")
only_lora_path, only_lora_inf_time, only_lora_vram = run_stable_diffusion_xl_only_lora()
trigger_words = lora_config.get("trigger_words", "LineAniAF, lineart")
only_lora_clip = compute_clip_score(only_lora_path, f"{core_prompt}, {trigger_words}")
only_lora_fid = compute_real_fid_score(only_lora_path)
only_lora_edges = compute_edge_density(only_lora_path)
print(f"  CLIP: {only_lora_clip} | FID: {only_lora_fid} | Speed: {only_lora_inf_time}s | VRAM: {only_lora_vram}MB | Edges: {only_lora_edges}%")

### Step 5.5: Benchmark SDXL + LoRA (Manga Style + Positive Prompts)
Runs the recommended high-end configuration, loading SDXL base, LoRA weights, and positive comic style descriptors.

In [ ]:
print(" [Part 5/5] Benchmarking SDXL + LoRA with Full Style Prompts...")
sdxl_lora_path, sdxl_lora_inf_time, sdxl_lora_vram = run_stable_diffusion_xl_with_lora()
sdxl_lora_clip = compute_clip_score(sdxl_lora_path, bench_prompt)
sdxl_lora_fid = compute_real_fid_score(sdxl_lora_path)
sdxl_lora_edges = compute_edge_density(sdxl_lora_path)
print(f"  CLIP: {sdxl_lora_clip} | FID: {sdxl_lora_fid} | Speed: {sdxl_lora_inf_time}s | VRAM: {sdxl_lora_vram}MB | Edges: {sdxl_lora_edges}%")

### Step 5.6: Compile Matrix & Plot Comparison Charts
Synthesizes the metrics from all 5 parts, generates a comparative dataframe, and plots visual charts.

In [ ]:
# Compile comparison DataFrame
data = {
    "Configuration": [
        "Stable Diffusion v1.5",
        "SD 1.5 + LoRA",
        "Stable Diffusion XL (Base)",
        "Only LoRA (SDXL + No Prompts)",
        "SDXL + LoRA (With Prompts)"
    ],
    "CLIP Text Score": [sd15_clip, sd15_lora_clip, sdxl_clip, only_lora_clip, sdxl_lora_clip],
    "FID Score": [sd15_fid, sd15_lora_fid, sdxl_fid, only_lora_fid, sdxl_lora_fid],
    "Inference Time (sec)": [sd15_inf_time, sd15_lora_inf_time, sdxl_inf_time, only_lora_inf_time, sdxl_lora_inf_time],
    "Peak VRAM (MB)": [sd15_vram, sd15_lora_vram, sdxl_vram, only_lora_vram, sdxl_lora_vram],
    "Edge Density (%)": [sd15_edges, sd15_lora_edges, sdxl_edges, only_lora_edges, sdxl_lora_edges]
}

df = pd.DataFrame(data)
print("\n Comparative Evaluation Matrix:")
display(df)

# Plotting comparisons
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
df.plot(x="Configuration", y="CLIP Text Score", kind="bar", ax=axes[0,0], color="skyblue", rot=30)
axes[0,0].set_title("CLIP Text Score (Higher is Better)")
df.plot(x="Configuration", y="FID Score", kind="bar", ax=axes[0,1], color="salmon", rot=30)
axes[0,1].set_title("FID Score (Lower is Better)")
df.plot(x="Configuration", y="Inference Time (sec)", kind="bar", ax=axes[1,0], color="lightgreen", rot=30)
axes[1,0].set_title("Inference Speed (Lower is Better)")
df.plot(x="Configuration", y="Peak VRAM (MB)", kind="bar", ax=axes[1,1], color="orchid", rot=30)
axes[1,1].set_title("Peak VRAM Usage (Lower is Better)")
plt.tight_layout()
plt.show()

### Step 5.1: Composite Recommendation & Model Selector
Computes a composite mathematical score to recommend the best model matching quality & resource constraints.

In [ ]:
max_time = df["Inference Time (sec)"].max()
max_vram = df["Peak VRAM (MB)"].max()
max_fid = df["FID Score"].max()

# Normalized Composite Score calculation
df["Composite Score"] = (
    0.3 * df["CLIP Text Score"] +
    0.3 * (1.0 - df["FID Score"] / (max_fid if max_fid > 0 else 1.0)) +
    0.2 * (1.0 - df["Inference Time (sec)"] / (max_time if max_time > 0 else 1.0)) +
    0.2 * (1.0 - df["Peak VRAM (MB)"] / (max_vram if max_vram > 0 else 1.0))
)

best_idx = df["Composite Score"].idxmax()
recommended_config = df.loc[best_idx, "Configuration"]
print(f" Recommended Configuration: {recommended_config} (Score: {df.loc[best_idx, 'Composite Score']:.3f})")

choice_map = {
    "Stable Diffusion XL (Base)": 1,
    "Stable Diffusion v1.5": 2,
    "SD 1.5 + LoRA": 2,
    "Only LoRA (SDXL + No Prompts)": 3,
    "SDXL + LoRA (With Prompts)": 3
}
default_choice = choice_map.get(recommended_config, 3)

# Select configuration programmatically or via user input prompt
print("\nSelect Model Configuration for final assets & panels generation:")
print("  1 = SDXL Base")
print("  2 = Stable Diffusion v1.5")
print("  3 = SDXL + LoRA (Manga Style Cel-shaded - Recommended)")

try:
    val = input(f"Enter model choice [1, 2, or 3, default={default_choice}]: ").strip()
    SELECTED_MODEL = int(val) if val else default_choice
except Exception:
    SELECTED_MODEL = default_choice

print(f"\n Confirmed selected configuration index: {SELECTED_MODEL}")

### Step 6: Generate Character Profile & Component Assets
Generates the character profile sheet (used as the anchor reference) along with 4 story component assets (secondary character, environment background, key prop).

In [ ]:
import subprocess, sys

print(f" Running generation pipeline using selected model config index: {SELECTED_MODEL}...")
if SELECTED_MODEL == 2:
    subprocess.run([sys.executable, "sd15_code/run_sd15_pipeline.py"], check=True)
elif SELECTED_MODEL == 3:
    subprocess.run([sys.executable, "lora_code/run_lora_pipeline.py"], check=True)
else:
    subprocess.run([sys.executable, "sdxl_code/run_sdxl_pipeline.py"], check=True)

from IPython.display import Image, display
import os

refs = [
    "outputs/characters/character_reference.png",
    "outputs/characters/character_reference_sd15.png",
    "outputs/characters/character_reference_sdxl_lora.png"
]
ref_found = next((r for r in refs if os.path.exists(r)), None)
if ref_found:
    print(f"\n Anchor Character Reference Profile Sheet ({ref_found}):")
    display(Image(ref_found))
else:
    print(" Reference character sheet image could not be loaded.")

### Step 6.1: Advanced Consistency Suite Validation
Evaluates character and asset visual coherence against the anchor reference using **8 specific mathematical & neural metrics**:

1. **Color Consistency**: Hue & Saturation HSV space correlation ($d(H_1, H_2)$)
2. **SSIM**: Structural Similarity Index (luminance, contrast, structural mapping)
3. **Canny Edge Density**: Compares drawing stroke active pixels ratio
4. **Art Style Gram Matrix**: Texture texture style correlations on Sobel gradients
5. **CLIP Image Similarity**: 512-dimensional semantic visual visual validation
6. **DINOv2 Feature Similarity**: High-fidelity spatial transformer representation alignment
7. **Offline Aesthetic Score**: Combines colorfulness, contrast, and sharpness locally
8. **Grayscale Correlation**: Legacy structure coefficient benchmark

In [ ]:
import sys, os
sys.path.append(os.getcwd())
sys.path.append('/content/indie_comic_pipeline')
sys.path.append('/content/Indie-Comic/indie_comic_pipeline')
import glob, pandas as pd
from utils.consistency_checker import get_consistency_checker

checker = get_consistency_checker()
refs = [
    "outputs/characters/character_reference.png",
    "outputs/characters/character_reference_sd15.png",
    "outputs/characters/character_reference_sdxl_lora.png"
]
ref_found = next((r for r in refs if os.path.exists(r)), None)
components = sorted(glob.glob("outputs/comics/component_*.png"))

if ref_found and components:
    print(f" Running full Consistency Suite with anchor: {ref_found}")
    checker.set_reference(ref_found)
    
    results = []
    for path in components:
        res = checker.check_consistency(path)
        results.append({
            "File": os.path.basename(path),
            "Overall Score": f"{res['score']:.3f}",
            "HSV Color Match": f"{res['color_score']:.3f}",
            "SSIM Structure": f"{res['ssim_score']:.3f}",
            "Edge Density Match": f"{res['edge_score']:.3f}",
            "Style Gram Match": f"{res['style_score']:.3f}",
            "Aesthetic Score": f"{res['aesthetic_score']:.3f}",
            "CLIP Semantic": f"{res['clip_img_score']:.3f}" if res['clip_img_score'] is not None else "N/A",
            "DINOv2 Structural": f"{res['dinov2_score']:.3f}" if res['dinov2_score'] is not None else "N/A",
            "Consistent": " Yes" if res['consistent'] else " No"
        })
        
    df_cons = pd.DataFrame(results)
    display(df_cons)
else:
    print(" Missing generated assets or anchor reference sheet. Run step 6 first.")

### Step 7: Generate Emotion-Aware Comic Panels for Storyboard Page
Renders each panel for the configured page, appending LLM dialogue expressions, and compiles them into final strip & grid layouts.

In [ ]:
import subprocess, sys

print(f" Rendering Storyboard Page {PAGE_TO_RENDER} panels layout...")
if SELECTED_MODEL == 2:
    subprocess.run([sys.executable, "sd15_code/generate_panels.py", "--page", str(PAGE_TO_RENDER)], check=True)
elif SELECTED_MODEL == 3:
    subprocess.run([sys.executable, "lora_code/generate_panels.py", "--page", str(PAGE_TO_RENDER)], check=True)
else:
    subprocess.run([sys.executable, "sdxl_code/generate_panels.py", "--page", str(PAGE_TO_RENDER)], check=True)

print(f"\n Page {PAGE_TO_RENDER} comic panel generation completed successfully!")

### Step 7.1: Visualise Comic Page Layout Grids

In [ ]:
import os, glob
from PIL import Image
import matplotlib.pyplot as plt

comics_dir = "outputs/comics"
layout_grids = glob.glob(os.path.join(comics_dir, f"page_{PAGE_TO_RENDER}_layout_*_grid.png"))

if layout_grids:
    for grid_path in sorted(layout_grids):
        print(f" Displaying Compiled Grid: {grid_path}")
        img = Image.open(grid_path)
        fig, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(img)
        ax.axis("off")
        plt.show()
else:
    # Fallback individual panels display
    panels = sorted(glob.glob(os.path.join(comics_dir, f"page_{PAGE_TO_RENDER}_panel_*.png")))
    if panels:
        n = len(panels)
        print(f" Displaying {n} individual panels:")
        fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
        if n == 1:
            axes = [axes]
        for ax, pf in zip(axes, panels):
            ax.imshow(Image.open(pf))
            ax.set_title(os.path.basename(pf), fontsize=8)
            ax.axis("off")
        plt.tight_layout()
        plt.show()
    else:
        print(" No generated layout grids or panel images found. Verify Step 7 outputs.")

### Step 7.2: Verify Comic Panels Character Consistency
Runs the Consistency Suite on the generated page panels to verify character visual integrity.

In [ ]:
import sys, os
sys.path.append(os.getcwd())
sys.path.append('/content/indie_comic_pipeline')
sys.path.append('/content/Indie-Comic/indie_comic_pipeline')
import glob, pandas as pd
from utils.consistency_checker import get_consistency_checker

checker = get_consistency_checker()
refs = [
    "outputs/characters/character_reference.png",
    "outputs/characters/character_reference_sd15.png",
    "outputs/characters/character_reference_sdxl_lora.png"
]
ref_found = next((r for r in refs if os.path.exists(r)), None)
panel_paths = sorted(glob.glob(f"outputs/comics/page_{PAGE_TO_RENDER}_panel_*.png"))

if ref_found and panel_paths:
    print(f" Verifying character consistency across panels utilizing anchor ref: {ref_found}")
    checker.set_reference(ref_found)
    
    results = []
    for path in panel_paths:
        res = checker.check_consistency(path)
        results.append({
            "Panel": os.path.basename(path),
            "Consistency score": f"{res['score']:.3f}",
            "HSV Color Match": f"{res['color_score']:.3f}",
            "SSIM structure": f"{res['ssim_score']:.3f}",
            "Edge Density Match": f"{res['edge_score']:.3f}",
            "Style Gram Match": f"{res['style_score']:.3f}",
            "Aesthetic Score": f"{res['aesthetic_score']:.3f}",
            "CLIP image match": f"{res['clip_img_score']:.3f}" if res['clip_img_score'] is not None else "N/A",
            "DINOv2 similarity": f"{res['dinov2_score']:.3f}" if res['dinov2_score'] is not None else "N/A",
            "Consistent": " Yes" if res['consistent'] else " No"
        })
    df_pan_cons = pd.DataFrame(results)
    display(df_pan_cons)
else:
    print(" No panel images found or reference sheet missing.")

### Step 8: Generate Custom Doodle Storyboard (Optional Test)
Runs the custom doodle generator script to create test storyboard frames and compile them into a grid sheet layout.

In [ ]:
print(" Generating custom doodle panels storyboard...")
!python generate_doodle_panels.py

import os
from PIL import Image
import matplotlib.pyplot as plt

doodle_grid = "outputs/comics/doodle_story_layout_grid.png"
if os.path.exists(doodle_grid):
    print(" Custom Doodle Layout Grid Sheet:")
    img = Image.open(doodle_grid)
    fig, ax = plt.subplots(figsize=(12, 12))
    ax.imshow(img)
    ax.axis("off")
    plt.show()
else:
    print(" Doodle storyboard layout grid image not found.")

### Step 8.5: Compile Comic Pages into PDF
Assembles all generated page layout grid sheets into a single, high-quality PDF book in the outputs directory.

In [ ]:
print(" Compiling comic book pages into PDF...")
!python compile_comic_pdf.py


### Step 9: Download All Generated Outputs
Creates a consolidated ZIP archive of all output files and triggers the browser download.

In [ ]:
import os, zipfile
from google.colab import files

ZIP_PATH = "/content/indie_comic_outputs.zip"
print(" Packaging outputs folder into ZIP archive...")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk("outputs"):
        for fname in fnames:
            fpath = os.path.join(root, fname)
            arcname = os.path.relpath(fpath, os.path.dirname("outputs"))
            zf.write(fpath, arcname)

size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f" ZIP created: {ZIP_PATH} ({size_mb:.1f} MB)")
print("⬇ Initiating browser download...")
files.download(ZIP_PATH)